# Lab 3 — The same desk, in ADK

**~25 minutes · nothing to fill in**

You built the Global Bank support desk as a crew in Labs 1 and 2. Now build it again in **Google ADK**
(Agent Development Kit). In Lab 4 you then compare two working versions of one desk, not two opinions
about frameworks.

ADK asks you to write some things that CrewAI hides: a **runner**, a **session**, and an **event
stream** that you read yourself. That is the trade-off. Watch for it.

**Words used in this lab**

- **Runner:** the ADK object that runs an agent and sends back events.
- **Event:** one step of a run, for example a tool call, a tool result or an answer. The runner sends
  them one at a time.
- **Session:** the conversation so far, stored by a **session service**.
- **Session state:** named values saved in the session. One agent can write a value, and a later agent
  can read it.
- **Sub-agent:** an agent that runs inside another agent, for example as one step in a fixed order.

## 1 · The model, and the first thing that breaks

⚠️ **ADK is the opposite of CrewAI in three ways.**

- It needs the `openai/` prefix on the model name.
- It takes `api_base`, not `base_url`.
- Its support for OpenAI-style models needs an extra package, `google-adk[extensions]`. Your sandbox
  already has it.

If you copy the CrewAI setup cell from Lab 1 into this notebook, it does not work.

In [ ]:
import os
from google.adk.models.lite_llm import LiteLlm

MODEL = os.environ["OPENAI_MODEL"]
BASE  = os.environ["OPENAI_BASE_URL"]
KEY   = os.environ["OPENAI_API_KEY"]

def model():
    return LiteLlm(
        model    = f"openai/{MODEL}",   # <- prefix needed, unlike CrewAI
        api_base = BASE,                # <- api_base, not base_url
        api_key  = KEY,
    )

print("model id:", model().model)

## 2 · A tool is just a function

You do not need a decorator. ADK reads the function's **signature and docstring** to build the tool
description that the model sees. So the type hints and the docstring are the interface. The lesson is
the same as in CrewAI. Only the mechanics are different.

An ADK tool usually returns a `dict`, not a string. A dict gives the model named fields.

In [ ]:
TICKETS = {
    "GB-T-4471": "My transfer of Rs 25,000 to my landlord failed twice, but my account was debited once.",
    "GB-T-4472": "I get 'invalid OTP' every time I log in on my new phone, since yesterday.",
    "GB-T-4473": "Please update my registered address. I have moved to Pune.",
    "GB-T-4474": "My savings account shows Rs 1,200 less than my passbook.",
}

# The four teams on the desk. They match the Global Bank services from Day 1.
TEAMS = ["Accounts", "Transactions", "Authentication", "Customer"]
TEAM_RULE = ("The category is the kind of problem, in a few words. "
             "The team must be one of: " + ", ".join(TEAMS) + ".")

def ticket_lookup(ticket_id: str) -> dict:
    """Return the text of a Global Bank customer support ticket by its id, e.g. GB-T-4471."""
    return {"ticket_id": ticket_id,
            "text": TICKETS.get(ticket_id, f"No ticket found with id {ticket_id}")}

print(ticket_lookup("GB-T-4471"))

## 3 · One agent, and the code to run it

Here ADK needs more setup than a crew. It helps to understand why.

- **`Runner`** runs the agent and sends back a stream of events.
- **`SessionService`** stores the conversation. ADK makes it an object that you can see and control.
- You **read the events yourself** and pick out the final answer.

For a one-agent job this is extra work. It starts to help you in sections 4 and 5.

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

triage = Agent(
    name="triage",
    model=model(),
    instruction=("Look up the ticket, then reply with 'Category: <x>' and 'Team: <y>' on separate lines. "
                 + TEAM_RULE),
    tools=[ticket_lookup],
)

session_service = InMemorySessionService()
runner = Runner(app_name="desk", agent=triage, session_service=session_service)

async def ask(text, agent_runner, session_id):
    """Run one turn and return (final_answer, prompt_tokens, candidate_tokens)."""
    await session_service.create_session(app_name="desk", user_id="u", session_id=session_id)
    msg = types.Content(role="user", parts=[types.Part(text=text)])
    prompt = candidates = 0
    final = None
    async for ev in agent_runner.run_async(user_id="u", session_id=session_id, new_message=msg):
        usage = getattr(ev, "usage_metadata", None)
        if usage:
            prompt     += usage.prompt_token_count or 0
            candidates += usage.candidates_token_count or 0
        if ev.is_final_response() and ev.content and ev.content.parts:
            final = ev.content.parts[0].text
    return final, prompt, candidates

answer, p, c = await ask("Classify ticket GB-T-4471.", runner, "s1")
print(answer)
print(f"\ntokens: prompt={p} candidates={c} total={p+c}")

> **You must use `await` here.** ADK's API is async first. `run_async` is the interface, and a
> notebook lets you `await` at the top level. ADK has no synchronous version that you could call by
> mistake. That is simpler than CrewAI, where `kickoff()` fails in a notebook.
>
> ADK calls the model's output tokens **candidates**. They are the same thing as CrewAI's completion
> tokens.

## 4 · Pass results between agents, by name

This is how ADK passes work from one agent to the next:

1. An agent saves its answer in **session state** under a name. The name is its `output_key`.
2. The next agent's instruction reads that value back with a `{placeholder}` of the same name.

In a crew, the previous task's output arrives as plain context. **In ADK, the hand-off has a name**, and
you can see it in the code.

In [ ]:
researcher = Agent(
    name="researcher", model=model(), tools=[ticket_lookup],
    instruction="Retrieve the ticket and list only the facts it contains. Do not interpret them.",
    output_key="facts",                       # <- saves the answer in session state
)

classifier = Agent(
    name="classifier", model=model(),
    instruction=("Using these facts:\n{facts}\n\n"          # <- reads it back by name
                 "Give 'Category: <x>' and 'Team: <y>' on separate lines. " + TEAM_RULE),
    output_key="triage",
)

drafter = Agent(
    name="drafter", model=model(),
    instruction=("Facts:\n{facts}\nTriage:\n{triage}\n\n"
                 "Write ONLY the reply the Global Bank customer will read - three sentences. "
                 "Do not introduce yourself, do not describe what you are doing, "
                 "never invent a timeline, and do not promise a refund or any action "
                 "that the facts do not support."),
)
print("three agents, two named hand-offs")

## 5 · Put them in order

`SequentialAgent` runs its sub-agents in order. The list is the wiring. This is the same idea as a
crew's task order. The difference is the data between the agents. Here it is state that you named, not
whatever the last agent happened to say.

In [ ]:
from google.adk.agents import SequentialAgent

desk = SequentialAgent(name="desk", sub_agents=[researcher, classifier, drafter])

desk_runner = Runner(app_name="desk", agent=desk, session_service=session_service)
answer, p, c = await ask("Handle ticket GB-T-4471.", desk_runner, "s2")
print(answer)
print(f"\ntokens: prompt={p} candidates={c} total={p+c}")

> ⚠️ **You will see a deprecation warning here.** ADK 2.9.1 says `SequentialAgent` is deprecated, and
> that you should use `Workflow` instead. But in this version you **cannot import `Workflow` from
> `google.adk.agents`**. The warning itself says `Workflow` "cannot yet be used as an LlmAgent
> sub-agent".
>
> So `SequentialAgent` is still the right choice today. ADK changes quickly. Some of its features are
> deprecated before their replacement is ready. That is real maintenance work for your team, so check
> the release notes whenever you upgrade ADK.
>
> You may also see `UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL` for each
> tool. It does not change the result.

## 6 · What to look at

- **The token total.** Write it down. Lab 4 compares it with the crew that did the same job.
- **The extra code.** The runner, the session and the event loop are more to set up than a crew. They
  also show you more when something goes wrong.
- **The hand-offs.** With `output_key` and `{facts}` you can point at exactly what the second agent
  received. In a crew, you trust CrewAI to pass the right context.
- **The reply.** If the drafter writes about itself ("Hello, I am the drafter...") and not to the
  customer, the fix is in the instruction, not in ADK.

### The other way to combine agents

`SequentialAgent` always runs the same order. Sometimes you want an agent to *decide* whether to ask
another agent. For that, wrap the other agent as a tool:

```python
from google.adk.tools import agent_tool
supervisor = Agent(name="supervisor", model=model(),
                   tools=[agent_tool.AgentTool(agent=classifier)])
```

This is ADK's version of delegation. Unlike a crew's manager, **you choose exactly which agents it can
reach.**

---

**Next:** Lab 4 runs this desk and the CrewAI desk side by side, and asks you to choose one.